# Analysis of CAE training results with encoded dimension 4

import os
os.system('python3 -m pip install --upgrade pip')
os.system('pip install tensorflow')Import classes

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
%matplotlib inline 

# Load trained model (& custom loss)

In [ ]:
from cae_model import convolutional_autoencoder, my_loss_func

original_dim = 9728
encoded_dim = 4
title = "CAE"

reconstructed_model = convolutional_autoencoder(original_dim, encoded_dim)
reconstructed_model.load_weights(f"./checkpoints/{title}_round04.weights.h5")    
reconstructed_model.compile(optimizer=Adam(), loss=my_loss_func, metrics=['mean_squared_error'])

# Create a reduced model from the original CAE model
# selecting only the encoder part, i.e. having the innermost ended layer as output layer
encoder = tf.keras.Model(inputs=reconstructed_model.get_layer(index=0).input,
                         outputs=reconstructed_model.get_layer('dense_encoded').output)

# Load data & process to obtain latent space values

This notebook has to be used after creating a test dataset with new waveforms with respect to those used for the training, and in a large quantity. The idea is to study the performances of the model as a function of the true signal peak in the waveforms, so a large dataset is necessary (it can be created using the application available in the "synthetic_waveforms" directory). 

In [ ]:
list_true_peak = []
list_encoded_wave = []
list_encoded_noise = []

# Load the file content
file_indices = range(0, 15)  # adjust as needed

for n in file_indices:
    filepath = f"../synthetic_waveforms/synthetic_waveforms_{n:02d}.npz"
    print(f"Opening file {filepath}")
    
    with np.load(filepath) as loaded:
        noise_wave  = loaded['noise']
        signal_wave = loaded['signal']
        true_peaks  = loaded['peak']
        #true_integrals = loaded['integral']
        #true_peak_positions = loaded['peak_position']

    wave = (noise_wave + signal_wave)[:, 125:-22-125, np.newaxis]
    noise_wave = noise_wave[:, 125:-22-125, np.newaxis]

    list_true_peak.append(true_peaks)
    list_encoded_wave.append(encoder(wave))
    list_encoded_noise.append(encoder(noise_wave))


    # Original dimension
    assert wave.shape[1] == original_dim, \
    f"File {n}: unexpected dimension {wave.shape[1]} != {original_dim}"

    del wave, noise_wave, signal_wave


# Build Pandas dataframes with complete information

In [ ]:
true_peaks = np.concatenate((list_true_peak), axis=0)
encoded_output_wave = np.concatenate((list_encoded_wave), axis=0)
encoded_output_noise = np.concatenate((list_encoded_noise), axis=0)

print(true_peaks.shape)
print(encoded_output_wave.shape)
print(encoded_output_noise.shape)

print(f"The ENCODED DIMENSION is {encoded_output_wave.shape[1]}")

waveDF = {
    "true_peak": true_peaks,
    "z0": encoded_output_wave[:,0],
    "z1": encoded_output_wave[:,1],
    "z2": encoded_output_wave[:,2],
    "z3": encoded_output_wave[:,3]
}

noiseDF = {
    "z0": encoded_output_noise[:,0],
    "z1": encoded_output_noise[:,1],
    "z2": encoded_output_noise[:,2],
    "z3": encoded_output_noise[:,3]
}

df_wave = pd.DataFrame(waveDF)
df_noise = pd.DataFrame(noiseDF)

del true_peaks, encoded_output_noise, encoded_output_wave

# Study the features of the latent space

In [ ]:
plt.figure(0, (9,9))

for i in range(encoded_dim):
    plt.subplot(2,2,i+1)
    plt.hist(df_wave[("z%s" % i)], bins=100, color='darkblue', label="wave set")
    plt.legend()
    plt.xlabel("latent_space[%s]" % (i))
    plt.yscale('log')
plt.show()

In [ ]:
plt.figure(0, (9,9))

for i in range(encoded_dim):
    plt.subplot(2,2,i+1)
    plt.hist(df_noise[("z%s" % i)], bins=100, color='green', label="noise set")
    plt.legend()
    plt.xlabel("latent_space[%s]" % (i))
    plt.yscale('log')
plt.show()

In [ ]:
for i in range(encoded_dim):
    plt.figure(i, (9,3))
    plt.hist(df_wave[("z%s" % i)], bins=150, range=[-0.25,0.25], color='darkblue', label="wave set")
    plt.hist(df_noise[("z%s" % i)], bins=150, range=[-0.25,0.25], color='green', alpha=0.75, label="noise set")
    plt.legend()
    plt.xlabel("latent_space[%s]" % (i))
    plt.yscale('log')
    plt.show()

In [ ]:
small_peak_cut = 0.01

for i in range(encoded_dim):
    plt.figure(i, (9,3))
    plt.hist(df_wave[("z%s" % i)], bins=150, range=[-0.25,0.25], color='darkblue', label="wave set")
    plt.hist(df_wave[df_wave["true_peak"]<small_peak_cut][("z%s" % i)], 150, range=[-0.25,0.25], color='darkorange', label=f"small peak (<{small_peak_cut})");
    plt.hist(df_noise[("z%s" % i)], bins=150, range=[-0.25,0.25], color='green', alpha=0.65, label="noise set")
    plt.legend()
    plt.xlabel("latent_space[%s]" % (i))
    plt.yscale('log')
    plt.show()

# Garage identification and definition of the intervals

In [ ]:
# Obtain the distribution of the latent space parameters for the noise-only dataset
z0 = df_noise["z0"]
z1 = df_noise["z1"]
z2 = df_noise["z2"]
z3 = df_noise["z3"]

In [ ]:
# Calculate the 3-sigma interval around the median
# using the numpy percentiles

half_tail_perc = (100-99.73)/2  # 3-sigma = 99.73%

z0_params = np.percentile(z0,(half_tail_perc,50,100-half_tail_perc))
z1_params = np.percentile(z1,(half_tail_perc,50,100-half_tail_perc))
z2_params = np.percentile(z2,(half_tail_perc,50,100-half_tail_perc))
z3_params = np.percentile(z3,(half_tail_perc,50,100-half_tail_perc))

params = np.stack( (z0_params,z1_params,z2_params,z3_params), axis=0)
print(params)

# Results of the methodology

In [ ]:
# Apply the garage cut to the noise-only dataset
# This gives an estimation of the false positives, i.e. events 
# that do not have a signal but are tagged as signals
# The percentage of this events should be around ~1% 
# given how we defined the garage

noise_cut_z0 = (df_noise["z0"]>params[0,0]) & (df_noise["z0"]<params[0,2])
noise_cut_z1 = (df_noise["z1"]>params[1,0]) & (df_noise["z1"]<params[1,2])
noise_cut_z2 = (df_noise["z2"]>params[2,0]) & (df_noise["z2"]<params[2,2])
noise_cut_z3 = (df_noise["z3"]>params[3,0]) & (df_noise["z3"]<params[3,2])

noise_tagged_as_noise = df_noise[(noise_cut_z0) & (noise_cut_z1) & (noise_cut_z2) & (noise_cut_z3)]
noise_tagged_as_signal = df_noise[~((noise_cut_z0) & (noise_cut_z1) & (noise_cut_z2) & (noise_cut_z3))]

print("Noise-only traces tagged as noise (i.e. inside the garage) :", len(noise_tagged_as_noise))
print("Noise-only traces tagged as signals (i.e. outside the garage) :", len(noise_tagged_as_signal))

In [ ]:
# Apply the garage cut to the train set

cut_z0 = (df_wave["z0"]>params[0,0]) & (df_wave["z0"]<params[0,2])
cut_z1 = (df_wave["z1"]>params[1,0]) & (df_wave["z1"]<params[1,2])
cut_z2 = (df_wave["z2"]>params[2,0]) & (df_wave["z2"]<params[2,2])
cut_z3 = (df_wave["z3"]>params[3,0]) & (df_wave["z3"]<params[3,2])

events_tagged_as_noise = df_wave[(cut_z0) & (cut_z1) & (cut_z2) & (cut_z3)]
events_tagged_as_signals = df_wave[~((cut_z0) & (cut_z1) & (cut_z2) & (cut_z3))]

print("Events tagged as noise-only (i.e. inside the garage) :", len(events_tagged_as_noise))
print("Events tagged as signals (i.e. outside the garage) :", len(events_tagged_as_signals))

In [ ]:
# Plot the results of the garage selection

plt.figure(1, (9,7))

plt.subplot(221)
plt.hist(np.log10(df_wave['true_peak']), bins=np.arange(-3,0,0.04), color='darkblue', label='wave set');
plt.hist(np.log10(events_tagged_as_signals['true_peak']), bins=np.arange(-3,0,0.04), color='magenta', label='tagged as signal', alpha=0.75);
plt.xlabel('log10(true peak)')
plt.xlim(-3,-1.5)
plt.legend()

plt.subplot(222)
plt.hist(df_wave['true_peak'], bins=100, range=(0,0.05), color='darkblue', label='wave set');
plt.hist(events_tagged_as_signals['true_peak'], bins=100, range=(0,0.05), color='magenta', label='tagged as signal', alpha=0.75);
plt.xlabel('true peak')
plt.yscale("log")
plt.legend()

plt.tight_layout()

In [ ]:
# Calculate at which peak amplitude the efficiency of the method 
# in tagging actual signals drops below 99%.

# Since the signals have been generated with an amplitude uniform in log10 of the peak height, 
# the study is performed dividing the training dataset in log10 equal bins

bin_width = 0.02
bin_edges = np.arange(-2.80,0,bin_width)
bin_centers = bin_edges + bin_width/2

count_tot, _ = np.histogram(np.log10(df_wave['true_peak']), bins=bin_edges, density=False)
count_tag, _ = np.histogram(np.log10(events_tagged_as_signals['true_peak']), bins=bin_edges, density=False)

ratio = count_tag/count_tot

for r,c,n1,n2 in zip(ratio, bin_centers, count_tag, count_tot):
    
    if(r >= 0.9):
        print("For (true) peak in log10 interval : [ %.3f, %.3f ]" % (c-bin_width/2,c+bin_width/2) )
        print("center corresponding to = %.5f --> r = %.4f ( %d / %d )" % (pow(10,c), r, n1, n2) )
    
    if(r >= 0.999):
        break


In [ ]:
from scipy.optimize import curve_fit
from scipy.special import erf

# Remove bins with zero counts to avoid division issues
valid = count_tot > 0
x_data = bin_centers[:-1][valid]
y_data = ratio[valid]

# Define the erf model
def erf_model(x, mu, sigma):
    return 0.5 + 0.5 * erf((x - mu) / sigma)

# Fit - p0 is the initial guess (from the previous hardcoded values)
popt, pcov = curve_fit(erf_model, x_data, y_data, p0=[-2.25, 0.15])
perr = np.sqrt(np.diag(pcov))

print(f"mu    = {popt[0]:.4f} +/- {perr[0]:.4f}  -->  peak = {10**popt[0]:.5f}")
print(f"sigma = {popt[1]:.4f} +/- {perr[1]:.4f}")

# Plot
plt.figure(figsize=(5, 4))
plt.plot(x_data, y_data, '.', label="bin-by-bin result")

x_fit = np.linspace(-3, 0, 300)
plt.plot(x_fit, erf_model(x_fit, *popt), label=f"fit: μ={popt[0]:.3f}, σ={popt[1]:.3f}")

plt.xlabel("log10(true peak)")
plt.ylabel("tagging efficiency")
plt.legend()
plt.show()